# Seurat to AnnData Conversion

**Assembly of AnnData Objects from Raw Spatial Transcriptomics Data**

This notebook details the process of assembling an `AnnData` (*.h5ad*) object from raw spatial transcriptomics data files. It covers the loading of count matrices and metadata, followed by their integration into a single `AnnData` structure suitable for downstream analysis. The notebook ensures data consistency, proper annotation, and storage in the standardized *.h5ad* format, facilitating efficient and reproducible analysis of spatial transcriptomics datasets.

In [ ]:
# ---------------
#     Imports
# ---------------

from datetime import datetime
from os import makedirs
import os.path as path

import anndata as ad
import pandas as pd
from scipy.io import mmread
from scipy.sparse import csr_matrix

## Run Information

In [ ]:
# Input directory
data_dir = "../output/data/Seurat_to_h5ad"

# Output directories
run_dir = f"run_{datetime.now().strftime('%Y-%m-%d_%H-%M')}"
out_dir = f"../output/data/h5ad_objects/{run_dir}"

# Ensure the output directory exists
makedirs(out_dir, exist_ok=True)

print(f"📂 Input Seurat data from: {data_dir}\n💾 Outputs will be saved in: {out_dir}")

📂 Input Seurat objects from: ../data/Seurat_to_h5ad
💾 Outputs will be saved in: ../output/data/h5ad_objects/run_2025-06-04_13-46


## Assemble Object

The `AnnData` object is built around a count matrix, which is often represented in a sparse representation.  
This means that only RNA counts greater than 0 are stored as actual values.

In [ ]:
# Load sparse count matrix from .mtx
sparse_matrix = mmread(path.join(data_dir, "Seurat_counts.mtx"))
sparse_matrix

<COOrdinate sparse matrix of dtype 'int64'
	with 16181234 stored elements and shape (970, 67785)>

Convert the matrix to the `CSR` format to improve performance.  
Improves run time since most operations are run on samples instead of features.  

&rarr; `Seurat` matrix needs to be transposed (other orientation)

In [5]:
# Convert to CSR format
csr_matrix = csr_matrix(sparse_matrix)
csr_matrix_t = csr_matrix.transpose().tocsr()
csr_matrix_t

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 16181234 stored elements and shape (67785, 970)>

Row and column names must be loaded separately because they are not saved in the `MMatrix object`.

Important:  
- AnnData row names are Seurat column names
- Same for AnnData column names and Seurat row names

In [ ]:
# Load row and column names from .csv
row_names = pd.read_csv(path.join(data_dir, "Seurat_col_names.csv"), header=None)
col_names = pd.read_csv(path.join(data_dir, "Seurat_row_names.csv"), header=None)

Metadata is stored in a matrix that can be initialized to the `obs` parameter during creation.

In [ ]:
# Load metadata from "csv"
obs_data = pd.read_csv(path.join(data_dir, "Seurat_metadata.csv"))

# Add row names as column
# Column can be set as index
obs_data["sample_names"] = row_names[0].values
obs_data = obs_data.set_index("sample_names")
obs_data.index.name = None  # Remove index title

# Rename columns
obs_data = obs_data.rename(
    columns={
        "nCount_Nanostring": "counts_per_cell",
        "nFeature_Nanostring": "non_zero_features",
        "cell_ID": "cell_id",
        "Slide_name": "slide",
    }
)

obs_data

,counts_per_cell,non_zero_features,cell_id,disease_state,slide,tissue,fov
liver1_3_1,450,181,c_1_1_3,healthy,S2,liver1,1_liver1
liver1_4_1,780,241,c_1_1_4,healthy,S2,liver1,1_liver1
liver1_5_1,752,260,c_1_1_5,healthy,S2,liver1,1_liver1
liver1_7_1,725,323,c_1_1_7,healthy,S2,liver1,1_liver1
liver1_8_1,395,142,c_1_1_8,healthy,S2,liver1,1_liver1
...,...,...,...,...,...,...,...
liver2_531_45,579,222,c_2_45_531,cirrhosis,S3,liver2,45_liver2
liver2_532_45,1015,270,c_2_45_532,cirrhosis,S3,liver2,45_liver2
liver2_533_45,954,264,c_2_45_533,cirrhosis,S3,liver2,45_liver2
liver2_534_45,2247,418,c_2_45_534,cirrhosis,S3,liver2,45_liver2


Compute if a gene (feature) has zero expression across all samples and save that as metadata.

In [8]:
# Convert to CSC to improve computing
csc_matrix = csr_matrix_t.tocsc()
csc_matrix

<Compressed Sparse Column sparse matrix of dtype 'int64'
	with 16181234 stored elements and shape (67785, 970)>

In [10]:
# Initialize data frame
var_data = pd.DataFrame(index=col_names[0].values)

# Sum counts per gene
counts_per_gene = csc_matrix.sum(axis=0)

var_data["counts_per_gene"] = counts_per_gene.A1
var_data

,counts_per_gene
Abca2,18947
Abi1,48137
Abi2,9686
Abl1,16608
Ace2,5576
...,...
NegPrb6,4902
NegPrb7,8887
NegPrb8,4277
NegPrb9,2622


In [13]:
adata = ad.AnnData(X=csr_matrix_t, obs=obs_data, var=var_data)
adata

AnnData object with n_obs × n_vars = 67785 × 970
    obs: 'counts_per_cell', 'non_zero_features', 'cell_id', 'disease_state', 'slide', 'tissue', 'fov'
    var: 'counts_per_gene'

## Sanity Checks

In [17]:
# Load sanity checks data frame
sanity_checks = pd.read_feather(path.join(data_dir, "sanity_checks.feather"))
sanity_checks

,n_rows,n_cols,count_sum,spotcheck_1,spotcheck_2,spotcheck_3,row_names,col_names,n_fovs,spotcheck_4,spotcheck_5,spotcheck_6
0,970,67785,65088926.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...","[Abca2, Abi1, Abi2, Abl1, Ace2, Acer3, Acta2, ...","[liver1_3_1, liver1_4_1, liver1_5_1, liver1_7_...",90,"[{'nCount_Nanostring': 450.0, 'nFeature_Nanost...","[{'nCount_Nanostring': 356.0, 'nFeature_Nanost...","[{'nCount_Nanostring': 579.0, 'nFeature_Nanost..."


In [ ]:
# Check rows in AnnData object
# ! n_rows and n_cols is inverted in AnnData compared to Seurat
n_cols = sanity_checks.n_cols[0]

# Validate that number of samples is identical
assert n_cols == adata.shape[0], f"Failed: Number of rows must be {n_cols}"

In [ ]:
n_rows = sanity_checks.n_rows[0]

# Validate that number of features is identical
assert n_rows == adata.shape[1], f"Failed: Number of cols must be {n_rows}"

In [ ]:
count_sum = sanity_checks.count_sum[0]

# Validate that sum of all counts is identical
assert count_sum == adata.X.sum(), f"Failed: Sum of counts must be {count_sum}"

In [54]:
# Validate that count matrix is identical
assert (
    adata.X[0:5, 0:5].toarray().flatten() == sanity_checks.spotcheck_1[0]
).all(), f"Failed: Values in count matrix [0:5, 0:5] are not identical"

assert (
    adata.X[33889:33894, 482:487].toarray().flatten() == sanity_checks.spotcheck_2[0]
).all(), f"Failed: Values in count matrix [33889:33894, 482:487] are not identical"

assert (
    adata.X[67780:67785, 965:970].toarray().flatten() == sanity_checks.spotcheck_3[0]
).all(), f"Failed: Values in count matrix [67780:67784, 965:969] are not identical"

In [62]:
# ! Row names are stored correctly
row_names = sanity_checks.row_names[0]

# Validate that row names are identical
assert (row_names == adata.var_names).all(), "Failed: Row names are not identical"

In [63]:
col_names = sanity_checks.col_names[0]

# Validate that row names are identical
assert (col_names == adata.obs_names).all(), "Failed: Column names are not identical"

In [68]:
# Number of FOVs must be 90 (45 per slide)
fovs = sanity_checks.n_fovs[0]

# Validate that number of FOVs are identical
assert fovs == len(
    adata.obs["fov"].unique()
), f"Failed: Number of unique FOVs must be {fovs}"

In [106]:
# Create Seurat metadata frame
seurat = pd.DataFrame(list(sanity_checks.spotcheck_4[0]))
seurat = list(range(seurat.shape[1]))  # Drop column names

# Create AnnData metadata frame
anndata = adata.obs.reset_index(drop=True).iloc[0:5]  # Drop row names
anndata = list(range(anndata.shape[1]))  # Also drop column names

# Verify that metadata is identical
assert seurat == anndata, "Failed: Metadata in range [0:5] is not identical"

In [105]:
seurat = pd.DataFrame(list(sanity_checks.spotcheck_5[0]))
seurat = list(range(seurat.shape[1]))

anndata = adata.obs.reset_index(drop=True).iloc[33889:33894]
anndata = list(range(anndata.shape[1]))

assert seurat == anndata, "Failed: Metadata in range [33889:33894] is not identical"

In [104]:
seurat = pd.DataFrame(list(sanity_checks.spotcheck_6[0]))
seurat = list(range(seurat.shape[1]))

anndata = adata.obs.reset_index(drop=True).iloc[67780:67785]
anndata = list(range(anndata.shape[1]))

assert seurat == anndata, "Failed: Metadata in range [67780:67785] is not identical"

## Save Object

Save `raw` counts in separate layer.

In [109]:
adata.layers["raw"] = adata.X

In [110]:
adata

AnnData object with n_obs × n_vars = 67785 × 970
    obs: 'counts_per_cell', 'non_zero_features', 'cell_id', 'disease_state', 'slide', 'tissue', 'fov'
    var: 'counts_per_gene'
    layers: 'raw'

In [ ]:
adata.write(path.join(out_dir, "liver_merged.h5ad"))